# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
meta_obj = dataset.metadata
print(f"{meta_obj.name}: {meta_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their IDs
print('Available record sets and fields:')
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet name: {getattr(record_set, 'name', '')}")
    print(f"  @id: {getattr(record_set, '@id', '')}")
    print(f"  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - {getattr(field, 'name', '')} (@id: {getattr(field, '@id', '')}, dataType: {getattr(field, 'data_type', '')})")
    print()

## 3. Data Extraction
Load data from the available record sets into DataFrames for analysis. Here, all relevant record sets (`@id`) are extracted, and you can select fields using their specific `@id` values for further work.

In [ ]:
# Identify the available record sets
record_sets = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', '') for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Pick the first populated record_set for demo
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No records could be loaded from the dataset.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

All field references use their Croissant `@id` for unambiguous mapping.

In [ ]:
import numpy as np

# For demonstration, select a numeric field from the first available DataFrame
if dataframes:
    df = dataframes[main_record_set_id]
    # Display all columns
    print('Available columns:', df.columns.tolist())
    # Attempt to identify a likely numeric column via Croissant field data types
    # Get mapping of @id to field info for the chosen record set
    record_set_obj = None
    for rs in dataset.metadata.record_sets:
        if getattr(rs, '@id', '') == main_record_set_id:
            record_set_obj = rs
            break
    field_id_to_type = {}
    field_id_to_name = {}
    if record_set_obj is not None and hasattr(record_set_obj, 'fields'):
        for field in record_set_obj.fields:
            field_id = getattr(field, '@id', '')
            field_id_to_type[field_id] = getattr(field, 'data_type', '')
            field_id_to_name[field_id] = getattr(field, 'name', '')

    # Try to choose a numeric field based on data type
    numeric_candidates = [fid for fid, dtype in field_id_to_type.items() if dtype in ['Float', 'Integer', 'Number'] and fid in df.columns]
    # Fallback: use string search on column names
    if not numeric_candidates:
        for col in df.columns:
            if any(s in col.lower() for s in ['amount', 'quant', 'value', 'count', 'mw', 'weight', 'abundance', 'score', 'intensity']):
                numeric_candidates.append(col)

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        # If field_id_to_name gives an English name, show it
        print(f"Field name: {field_id_to_name.get(numeric_field_id, numeric_field_id)}")
        # Attempt conversion
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.75)  # top quartile for illustration
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (Q3):")
        print(filtered_df[[numeric_field_id]].head())
        # Normalization (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Attempt grouping by a candidate field
        # Look for a field likely categorical
        group_field_candidates = []
        for fid, dtype in field_id_to_type.items():
            if dtype not in ['Float', 'Integer', 'Number'] and fid in filtered_df.columns and dtype:
                group_field_candidates.append(fid)
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping by {group_field} (field name: {field_id_to_name.get(group_field, group_field)})")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame('mean_value')
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field available for analysis in this record set.")
else:
    print('No DataFrames available for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if dataframes and 'numeric_field_id' in locals():
    # Histogram
    plt.figure(figsize=(6,4))
    df[numeric_field_id].dropna().hist(bins=30)
    plt.xlabel(field_id_to_name.get(numeric_field_id, numeric_field_id))
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {field_id_to_name.get(numeric_field_id, numeric_field_id)}')
    plt.show()
    # If we found a group_field, display boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(8,5))
        filtered_df.boxplot(column=numeric_field_id, by=group_field)
        plt.title(f'{field_id_to_name.get(numeric_field_id, numeric_field_id)} by {field_id_to_name.get(group_field, group_field)}')
        plt.suptitle('')
        plt.xlabel(field_id_to_name.get(group_field, group_field))
        plt.ylabel(field_id_to_name.get(numeric_field_id, numeric_field_id))
        plt.show()
else:
    print('Visualization skipped: No numeric field found for plotting.')

## 6. Conclusion
This notebook provided an end-to-end exploration pipeline for the FAIR^2 Croissant dataset [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells].

**Key findings:**
- Metadata and available record sets were inspected using unique `@id` references.
- Data was extracted to DataFrames and processed entirely via Croissant field `@id` identifiers.
- A numeric field was analyzed, filtered, normalized, and visualized, providing a reproducible workflow for further work.

To adapt for other projects, update the dataset URL and select record set and field `@id` values as appropriate.